In [1]:
import sys

sys.path.append("..")

In [2]:
import os
import copy
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from os.path import join as pjoin

from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset
from torch import nn

from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

from utils.misc import load_config, load_model_from_checkpoint
from datasets.data_preparation import prepare_data
from utils.simulated_reward_strategy import get_simulated_rewards_as_labels_for_reward_model

/home/nasim/anaconda3/envs/ML/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
exp_path = '../results/policy/dl.vit/2023-08-21 18Hr 50Min 03Sec IST+0530'

config = load_config('.', exp_path, 'hyperparameters.yaml')

In [4]:
trainer_config = config['trainer']
data_config = config['data']

model_name = config['model']['__name__']

checkpoint_name = trainer_config['checkpoint_name']
batch_size = trainer_config['batch_size']

patch_size = data_config['patch']['patch_size']
lithology_classes = data_config['lithology_classes']

config['root'] = '..'

In [8]:
config['trainer']['device'] = 'cpu'
device = trainer_config['device']
config['model']['use_lora'] = False

In [6]:
if model_name.startswith('dl'):
   if model_name.endswith('vit'):
        from model.vit import build_model
   elif model_name.endswith('ann'):
        from model.ann import build_model
    

model = build_model(config)
model, _ = load_model_from_checkpoint(model, checkpoint_name, exp_path, device)

Building the model...
Loading model from checkpoint...


In [9]:
_, x_val, _, y_val, _ = prepare_data(config)

Preparing the data...


Creating Patches: 100%|██████████| 943/943 [00:16<00:00, 55.84it/s]


Number of classes: 6 and shape of x_train: torch.Size([8046, 150, 6])


In [10]:
valdataset = TensorDataset(x_val, y_val)
valloader = DataLoader(valdataset, batch_size=batch_size, shuffle=False)

# invert key as value and value as key
lithology_names = {v: k for k, v in lithology_classes.items()}

In [11]:
inp, gt = next(iter(valloader))

In [12]:
model.device

'cpu'

In [13]:
model = model.to(model.device)

In [14]:
model.eval()
out = model(inp.to(model.device))

In [15]:
_, attn, o = out

ValueError: too many values to unpack (expected 3)

In [ ]:
plt.imshow(o[0, :, :].detach().cpu().numpy(), cmap='gray')
plt.colorbar()

In [ ]:
[2]*10

In [ ]:
gt[32]*10

In [ ]:
attn_value = attn[0, 0, :, :].detach().cpu().numpy()
gt_values = gt[0, :].cpu().numpy()

In [ ]:
plt.imshow(attn_value, cmap='gray')
plt.plot(gt_values*10, 'red', linewidth=3)
plt.colorbar()
plt.savefig('attn.png')